# S9 — Pilot Training (5 epochs, FocalLoss)\nQuick sanity check: train 5 epochs with FocalLoss to verify the model learns to segment tumors.

## Setup: Imports & GPU Check

In [ ]:
import sys, json, os
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import torch
print(f'Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## Configuration

In [ ]:
EPOCHS = 5
BATCH_SIZE = 4
LR = 1e-3
SEED = 42
OUTPUT_DIR = Path('../models/s9_pilot')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output: {OUTPUT_DIR}')
print(f'Epochs: {EPOCHS}, Batch: {BATCH_SIZE}, LR: {LR}')

## Data Preparation

In [ ]:
from src.config import DEVICE, print_device_info
from src.utils import set_seed
from src.data_loader import DatasetConfig, DataPathManager, VolumeWiseSplitter, create_2d_dataloaders
from src.preprocessing import PreprocessingTransform, AugmentedPreprocessingTransform, CLAHEProcessor
from src.models import create_model, count_params
from src.trainer import Trainer

set_seed(SEED); print_device_info()

path_manager = DataPathManager()
volume_index = path_manager.build_index()

splitter = VolumeWiseSplitter()
splits = splitter.load_splits(DatasetConfig.SPLITS_DIR)
print(f'Loaded splits: train={len(splits["train"])}, val={len(splits["val"])}, test={len(splits["test"])}')

clahe = CLAHEProcessor(clip=2.0, grid=(8, 8))
transform_train = AugmentedPreprocessingTransform(
    target_size=(256, 256), hu_low=-100, hu_high=400, clahe=clahe)
transform_val = PreprocessingTransform(
    target_size=(256, 256), hu_low=-100, hu_high=400)

train_loader, val_loader, test_loader = create_2d_dataloaders(
    volume_index, splits['train'], splits['val'], splits['test'],
    batch_size=BATCH_SIZE, transform_train=transform_train, transform_val=transform_val)
print(f'Batches: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}')

## Model & Trainer Setup

In [ ]:
model = create_model('mobilenetv2_unet', in_channels=1, out_channels=1, pretrained=True)
model = model.to(DEVICE)
print(f'Parameters: {count_params(model):,}')

from src.config import PHASE4_RESEARCH_CONFIG
train_config = dict(PHASE4_RESEARCH_CONFIG)
train_config['use_focal'] = True

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=train_config,
    learning_rate=LR,
    num_epochs=EPOCHS,
    mixed_precision=True,
    output_dir=str(OUTPUT_DIR))
print('Trainer ready.')

## Train (5 epochs — should take ~5 min)

In [ ]:
print('Training with FocalLoss...')
trainer.train(warmup_epochs=0)
print(f'Best val dice: {trainer.best_val_dice:.4f}')

## Save History & Evaluate on Test Set

In [ ]:
with open(OUTPUT_DIR / 'history.json', 'w') as f:
    json.dump(trainer.history, f, indent=2)

test_metrics = trainer.evaluate(test_loader)
print(f'Test Dice: {test_metrics["dice"]:.4f}, IoU: {test_metrics["iou"]:.4f}')
with open(OUTPUT_DIR / 'test_metrics.json', 'w') as f:
    json.dump(test_metrics, f, indent=2)
print('Done.')

## Results Summary

In [ ]:
import pandas as pd
hist = trainer.history
df = pd.DataFrame({'epoch': range(1, EPOCHS+1), 'train_dice': hist['train_dice'], 'val_dice': hist['val_dice']})
print(df.to_string(index=False))
print(f'\nBest val_dice: {max(hist["val_dice"]):.4f} at epoch {1+hist["val_dice"].index(max(hist["val_dice"]))}')